In [1]:
%pip install pandas
%pip install requests
%pip install alpaca-py
%pip install pytz
%pip install pytickersymbols
%pip install pyarrow
%pip install yfinance
%pip install lxml
%pip install beautifulsoup4


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[noti

In [ ]:
from alpaca.data.timeframe import TimeFrame
from alpaca.data.requests import StockBarsRequest
from alpaca.data import StockHistoricalDataClient
from datetime import datetime
from matplotlib import ticker
import pandas as pd
import os
import pyarrow
import yfinance as yf

stock_client = StockHistoricalDataClient("PKDBAP3S6DPDWEUTLD77R256C6", "DVKUFxa3UNtu4yJuWXFWyuzGAEaqCkSgZvJvgBsectr1")


from alpaca.broker import client
from alpaca.trading.requests import GetAssetsRequest

#Note: I learned only AFTER writing this, you need seperate api keys for the broker client
# Get those here: https://broker-app.alpaca.markets/dashboard

broker_client = client.BrokerClient("CKZXMIOPRAG37W7T53KW5F2YOW", "8hA9KKQhF7ZWnFdb78Pb2dqenm6hBC1NUDqdUnHvzTH")

request_params = GetAssetsRequest(
    status="active",
    asset_class="us_equity",
    exchange="NYSE"
)

nyse_assets = broker_client.get_all_assets(request_params)
nasdaq_assets = broker_client.get_all_assets(GetAssetsRequest(status="active", asset_class="us_equity", exchange="NASDAQ"))

print(f"Active NYSE assets: {len(nyse_assets)}")
print(f"Active NASDAQ assets: {len(nasdaq_assets)}")

def fetch_symbols_from_response(nyse_assets, nasdaq_assets):
    #This is called list comprehension
    nyse_symbols = [asset.symbol for asset in nyse_assets if asset.status == "active" and asset.tradable]
    nasdaq_symbols = [asset.symbol for asset in nasdaq_assets if asset.status == "active" and asset.tradable]
    return nyse_symbols + nasdaq_symbols
symbols_list = fetch_symbols_from_response(nyse_assets, nasdaq_assets)
print(len(symbols_list), "Count of symbols fetched from broker API")


def find_top_1000_assets(symbols_list, stock_client):
    # This function will fetch the latest daily dollar volume for each symbol and return the top 1000 by that metric.
    bars = stock_client.get_stock_bars(StockBarsRequest(
        symbol_or_symbols=symbols_list,
        timeframe=TimeFrame.Day,
        start=datetime(2026, 1, 1),
        end=datetime(2026, 2, 1)
    ))
    bars_df = bars.df
    bars_df["dollar_volume"] = bars_df["close"] * bars_df["volume"]
    # Sort symbols by dollar volume and take top 1000
    top_1000 = bars_df.groupby("symbol")["dollar_volume"].sum().nlargest(1000)
    return top_1000

top_1000_symbols = find_top_1000_assets(symbols_list, stock_client)
#print("Top 1000 symbols by latest daily dollar volume:")
#print(top_1000_symbols)

def get_stock_data(ticker):

    def get_historical_data(ticker):
        formatted_request = StockBarsRequest(
            symbol_or_symbols=[ticker], 
            start=datetime(2025, 1, 1), # This could change right now it is a hardcoded value but in the future it could be dynamic 
            end=datetime(2026,2,24), # This could change right now it is a hardcoded value but in the future it could be dynamic 
            timeframe=TimeFrame.Minute
        )
        response = stock_client.get_stock_bars(formatted_request)
        #print(response)
        return response


    mydf = get_historical_data(ticker).df

    #print(mydf.columns) # This is just to check what columns we have in the DataFrame it also is basically a test to make sure the above works
    #print(mydf.describe())

    mydf = mydf.reset_index()
    mydf = mydf.set_index("timestamp")

    mydf = mydf.tz_convert("US/Central")
    mydf = mydf.between_time("08:30", "15:00")
    mydf = mydf[mydf.index.dayofweek < 5]

    market_open = pd.to_datetime("08:30").time()
    market_close = pd.to_datetime("15:00").time()


    full_timeline = pd.date_range(start=mydf.index.min(),end=mydf.index.max(), freq="1min", tz="US/Central")
    full_timeline = full_timeline[(full_timeline.time >= market_open) & (full_timeline.time <= market_close) & (full_timeline.dayofweek < 5)]

    mydf = mydf.reindex(full_timeline)
    mydf['symbol'] = ticker

    mydf[['open', 'high', 'low', 'close']] = mydf[['open', 'high', 'low', 'close']].ffill()
    mydf['volume'] = mydf['volume'].fillna(0)

    mydf = mydf.drop(columns=['trade_count','vwap'])

    mydf = mydf.reset_index()

    mydf = mydf.rename(columns={
        "index": "date",    
        "symbol": "ticker"
    })
    print(mydf.info())
    
    other_cols = [col for col in mydf.columns if col != 'ticker']

    new_order = ['ticker'] + other_cols

    mydf = mydf[new_order]

    #print(mydf.describe())
    return mydf


def get_1000_stock_info(ticker_list):

    all_dfs = []

    for ticker in ticker_list:
        df = get_stock_data(ticker)
        all_dfs.append(df)
        print(f"Finished {ticker}")
    
    final_df = pd.concat(all_dfs,ignore_index=True)

    return final_df



# This is temporary as its to test if my function works or not
#tempdf = get_1000_stock_info(ticker_list = top_1000_symbols.index.tolist()[:5]) # This [:5] at the end limits it to only the first 5 ticker symbols. When we want all thousand we just remove the [:5])
#print(tempdf)

#tempdf.to_parquet("1000Stocks.parquet")


import requests
from bs4 import BeautifulSoup
import yfinance as yf

def get_sp500_tickers():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    table = soup.find("table", {"id": "constituents"})
    tickers = []
    for row in table.find_all("tr")[1:]:
        ticker = row.find_all("td")[0].text.strip()
        tickers.append(ticker)
    return tickers

sp500tickers = get_sp500_tickers()
print(len(sp500tickers))
print(sp500tickers)

def find_other_500(symbols_list, sp500tickers, stock_client):
    stocks = symbols_list
    stocks = [symbol for symbol in symbols_list if symbol not in sp500tickers]


    bars = stock_client.get_stock_bars(StockBarsRequest(
        symbol_or_symbols=stocks,
        timeframe=TimeFrame.Day,
        start=datetime(2026, 1, 15),
        end=datetime(2026, 2, 15)
    ))
    bars_df = bars.df
    bars_df["dollar_volume"] = bars_df["close"] * bars_df["volume"]
    
    avg_dollar_volume = (bars_df.groupby(level="symbol")["dollar_volume"].mean())

    avg_price = (bars_df.groupby(level="symbol")["close"].mean())

    filtered = [
        s for s in avg_dollar_volume.index
        if avg_dollar_volume[s] >= 25_000_000
        and avg_price[s] > 10
    ]

    market_caps = {}
    for s in filtered:
        try:
            yf_symbol = s.replace(".", "-")
            info = yf.Ticker(yf_symbol).info
            market_caps[s] = info.get("marketCap", 0)
        except Exception:
            market_caps[s] = 0

    final_filtered = [s for s in filtered if market_caps.get(s, 0) >= 2_000_000_000]

    final_filtered = sorted(final_filtered, key=lambda s: avg_dollar_volume[s], reverse=True)[:497]

    if len(final_filtered) < 497:
        remaining = 497 - len(final_filtered)
        candidates = [s for s in avg_dollar_volume.index if s not in final_filtered]
        candidates_sorted = sorted(candidates, key=lambda s: avg_dollar_volume[s], reverse=True)
        final_filtered += candidates_sorted[:remaining]


    #print(f"After market cap filter: {len(final_filtered)} stocks")
    #print(final_filtered)


    return final_filtered




other_497 = find_other_500(symbols_list, sp500tickers, stock_client)
final_1000 = sp500tickers[:503] + other_497
#print(len(final_1000))
#print(final_1000)

pd.DataFrame({"ticker": final_1000}).to_parquet("final_1000_tickers.parquet")

tempdf = get_1000_stock_info(final_1000)
print(tempdf)

tempdf.to_parquet("FINAL1000Stocks.parquet")